<center>

# MINHA PRIMEIRA REDE NEURAL

</center>
</br>

- O objetivo deste notebook é orientar a implementação de uma rede neural feedforward simples;
- Essa rede recebe como entrada 4 inteiros, sendo eles comprimentos;
- A sáida deve ser se os comprimentos formariam um triângulo, um retângulo, ou nenhum destes.

---

**Exemplos:**

1. Triângulo equilátero (tem 3 lados iguais)
- entrada: [0, 1, 1, 1] 
- saída: [0] => triângulo

2. Triângulo isósceles (tem 3 lados, dos quais 2 são iguais)
- entrada: [5, 0, 5, 1] 
- saída: [0] => triângulo

3. Triângulo escaleno (tem 3 lados diferentes que satisfazem a condição de existência do triângulo)
- entrada: [3, 5, 0, 4] 
- saída: [0] => triângulo

4. Quadrado (4 lados iguais)
- entrada: [9, 9, 9, 9] 
- saída: [1] => retângulo

5. Retângulo (4 lados, com 2 pares iguais)
- entrada: [2, 7, 7, 2] 
- saída: [1] => retângulo

6. X (nenhum lado)
- entrada: [0, 0, 0, 0] 
- saída: [2] => Nenhum dos dois

7. X (um lado)
- entrada: [0, 1, 0, 0] 
- saída: [2] => Nenhum dos dois

8. X (dois lados)
- entrada: [0, 1, 0, 7] 
- saída: [2] => Nenhum dos dois

9. X (3 lados diferentes que não satisfazem a condição de existência do triângulo)
- entrada: [0, 2, 7, 10] 
- saída: [2] => Nenhum dos dois

10. X (4 lados, sem que 2 pares de lados sejam iguais)
- entrada: [1, 2, 3, 4] 
- saída: [2] => Nenhum dos dois
---

<center>

## 1. Importar bibliotecas necessárias

</center>
</br>

---

- O numpy será usado para implementar e interagir com vetores e matrizes;
- csv será usado para importar os dados de um arquivo.csv.

In [12]:
import numpy as np
import csv

- A função train_test_split é necessária para, dado uma amostra de dados rotulados, dividir parte desses dados para treinar o modelo e parte para testá-lo.

In [ ]:
from sklearn.model_selection import train_test_split

- O TensorFlow Keras será a API usada para construir e treinar a rede neural.
    - O Sequential() é usado para criar um modelo de forma sequencial, isto é, declarando camada por camada;
    - O Dense() será usado para criar cada camada, sendo elas de forma densa, ou seja, totalmente conectada com as outras;
    - O to_categorical() converte rótulos (saídaa) de classes (possíveis respostas) em one-hot encoding, formato necessário para redes neurais com saída categórica.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical

<center>

## 2. Gerar dados

</center>
</br>

<div align="justify">&emsp;&emsp;
Para que a rede neural aprenda de forma efetiva, são necessários grandes conjuntos de dados, isto é, problemas rotulados (resolvidos). Como o problema em questão é simples e pode ser resolvido de forma lógica, foi usado um gerador de dados para criar casos resolvidos do problema em questão.
</div></br>

---

- Funções para determinar o que são triângulos e retângulos.

In [13]:
def eh_triangulo(lados):
    lados = sorted(lados)
    return lados[0] == 0 and lados[1] + lados[2] > lados[3]

def eh_retangulo(lados):
    lados = sorted(lados)
    return lados[0] == lados[1] and lados[2] == lados[3] and lados[0] != lados[2]

- Função para gerar retângulos.
    - Se usar números aleatórios para gerar os dados, quase nenhum retângulo será gerado (probabilidade de números iguais é baixa), o que fará com que a rede neural não consiga identificar corretamente quadrados e retângulos. Logo, usa-se essa função para criar uma amostra boa e equilibarda de dados.

In [14]:
def gerar_retangulo():
    a = np.random.randint(1, 10)
    b = np.random.randint(1, 10)
    ordem_lados = np.random.randint(0, 6)
    opcoes = [[a, a, b, b], [b, b, a, a], [a, b, b, a],[b, a, a, b], [a, b, a, b], [b, a, b, a]]
    return opcoes[ordem_lados]

- Gerador de dados (criará o mesmo número de dados para cada classe).

Escolher a quantidade de dados a serem gerados por classe alterando o valor da variável ```qtd_por_classe```. O número de dados será 3x o valor da variável.

In [23]:
np.random.seed(42) # O Guia do Mochileiro das Galáxias
qtd_por_classe = 10000  
dados = []

# Criar triângulos
while len([d for d in dados if d[4] == 0]) < qtd_por_classe:
    lados = np.random.randint(0, 10, size=4)
    if eh_triangulo(lados):
        dados.append([*lados, 0])

# Criar retângulos
while len([d for d in dados if d[4] == 1]) < qtd_por_classe:
    dados.append([*gerar_retangulo(), 1])

# Criar exemplos que não são triângulos nem retângulos
while len(dados) < qtd_por_classe * 3:
    lados = np.random.randint(0, 10, size=4)
    if not eh_triangulo(lados) and not eh_retangulo(lados):
        dados.append([*lados, 2])

np.random.shuffle(dados)

arquivo = "dados.csv"

with open(arquivo, mode="w", newline="") as arquivo:
    escritor = csv.writer(arquivo)
    escritor.writerows(dados)

<center>

## 3. Preparar Dados

</center>
</br>

<div align="justify">&emsp;&emsp;
Uma vez que o conjunto de dados existe, deve-se importá-lo, e preparar os dados para uso efetivo e eficiente. Neste caso, isso significa separar entradas (features) e saídas (rótulos), escolher parte dos dados para treinar a rede e parte para testá-la e, por fim, transformar os rótulos em one-encoding para que a saída funcione. 
</div></br>

---

In [16]:
set_dados = np.loadtxt('dados.csv', delimiter=',')
print(np.shape(set_dados))

(9999, 5)


In [17]:
# Separar as entradas/features e a saída/rótulo
inp_dados = set_dados[:, :-1]
out_dados = set_dados[:, -1]

In [ ]:
# Separar parte dos dados para treinar o modelo e parte para testá-lo 
# train_test_split(entradas, saidas, test_size=porcentagem_para_teste, random_state=semente_do_randomizador)
inp_treino, inp_teste, out_treino, out_teste = train_test_split(inp_dados, out_dados, test_size=0.2, random_state=42) # 20% para teste 80% para treino

In [ ]:
# Transformar os rótulos em one-hot encoding (necessário para classificação categórica)
out_treino_cat = to_categorical(out_treino, num_classes=3)
out_teste_cat = to_categorical(out_teste, num_classes=3)

<center>

## 4. Criar Modelo

</center>
</br>

- Foi criado um modelo de 4 camadas (sendo uma de entrada e uma de saída), sendo elas declaradas sequencialmente e sendo fortemente conectadas entre si.

<div align="justify">&emsp;&emsp;
Para criar esse modelo, utilizou-se a classe `Sequential` da biblioteca `tensorflow.keras.models`. Essa classe permite empilhar camadas densamente conectadas em sequência, facilitando a construção de redes neurais. Cada camada foi adicionada por meio da função `Dense`, que define o número de neurônios e a função de ativação utilizada.
</div></br>

<div align="justify">&emsp;&emsp;
A primeira camada oculta tem 16 neurônios e usa a função de ativação **ReLU** (`relu`), que permite a introdução de não-linearidade na rede, melhorando a capacidade de aprendizado do modelo. A segunda camada oculta também possui 16 neurônios com ativação `relu`. Por fim, a camada de saída contém 3 neurônios e utiliza a função de ativação **Softmax** (`softmax`), que converte as saídas em probabilidades para classificação multiclasse.
</div></br>

<div align="justify">&emsp;&emsp;
Após a definição das camadas, o modelo foi compilado utilizando a função `compile()`. O otimizador escolhido foi o **Adam**, que ajusta automaticamente os pesos da rede para minimizar a função de perda. A função de perda utilizada foi **categorical crossentropy**, que mede a diferença entre a distribuição prevista pelo modelo e a distribuição real das classes, sendo apropriada para classificação multiclasse. A métrica de avaliação selecionada foi a **acurácia**, que mede a proporção de previsões corretas.
</div></br>

---

In [ ]:
# Criar a rede neural
model = Sequential([
    Dense(16, activation='relu', input_shape=(4,)),  # Camada oculta com 16 neurônios
    Dense(16, activation='relu'),  # Outra camada oculta
    Dense(3, activation='softmax')  # Camada de saída (3 classes)
])

# Compilar o modelo
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

<center>

## 5. Treinar modelo

</center>
</br>

<div align="justify">&emsp;&emsp;
O treinamento do modelo é realizado utilizando a função `fit()`, que ajusta os pesos da rede neural com base nos dados fornecidos. Esse processo é fundamental para que a rede aprenda padrões nos dados e consiga realizar previsões precisas.
</div></br>

<div align="justify">&emsp;&emsp;
O conjunto de treinamento (`inp_treino` e `out_treino_cat`) é passado como entrada para a função, permitindo que o modelo ajuste seus parâmetros ao longo das **50 épocas** (`epochs=50`). Cada época representa uma iteração completa sobre o conjunto de treinamento.
</div></br>

<div align="justify">&emsp;&emsp;
O parâmetro `batch_size=32` define que os dados serão divididos em lotes de 32 amostras antes de atualizar os pesos. Isso ajuda a otimizar a eficiência computacional e a estabilidade do treinamento, reduzindo variações abruptas no aprendizado.
</div></br>

<div align="justify">&emsp;&emsp;
Além disso, foi fornecido um conjunto de validação (`validation_data=(inp_teste, out_teste_cat)`). Esse conjunto é usado para medir o desempenho do modelo após cada época, permitindo identificar possíveis sinais de **overfitting** (quando o modelo memoriza os dados de treino e não generaliza bem para novos dados).
</div></br>

<div align="justify">&emsp;&emsp;
Ao longo do treinamento, o modelo ajusta seus pesos de acordo com a função de perda e o otimizador definidos anteriormente, buscando minimizar os erros e melhorar a precisão da rede neural.
</div></br>

---

In [ ]:
# Treinar o modelo
model.fit(inp_treino, out_treino_cat, epochs=50, batch_size=32, validation_data=(inp_teste, out_teste_cat))

<center>

## 6. Testar modelo

</center>
</br>

---

In [ ]:
import numpy as np

# Função para pedir entrada do usuário e converter para um array
def inserir_comprimentos():
    print("\nDigite os 4 comprimentos (separados por espaço) ou '0 0 0 0' para sair:")
    entrada = input()  # Lê uma linha de entrada do usuário
    valores = list(map(int, entrada.split()))  # Converte para lista de inteiros

    if len(valores) != 4:
        print("Por favor, insira exatamente 4 valores.")
        return inserir_comprimentos()  # Tenta novamente
    
    return np.array([valores])  # Retorna um array 2D (necessário para o modelo)

# Fazer a predição para a entrada manual
while True:
    entrada_manual = inserir_comprimentos()

    # Se os quatro valores forem 0, encerra o programa
    if np.all(entrada_manual == 0):
        break

    pred = model.predict(entrada_manual)  # Faz a predição
    resposta = np.argmax(pred)  # Pega a classe com maior probabilidade

    # Interpretar a saída
    if resposta == 0:
        print("Triângulo")
    elif resposta == 1:
        print("Retângulo")
    else:
        print("NULL")